# T2V-1K · upgrade test: AMD TokenFactory reads the classes, photos teach the rows

The diagnostics say the classifier rows of never-seen classes are the bottleneck (59.1% with predicted rows vs 71.6% with fine-tuned rows on the same generated body). Here the text → row map is trained on the photos of the 900 seen classes, and its input is what AMD TokenFactory wrote about every class: the 5 visual descriptions per class, and a 60-attribute visual questionnaire (colour, texture, shape, parts, scene, kind; `t2v/attributes.py`). Every setting is chosen by cross-validation on seen classes; the official 100-task test splits are scored once, against the shipped T2V-1K, with ablations (names only, no TokenFactory; attributes only).

**Runtime → Change runtime type → T4 GPU**, then run the cell (if it ran before in this session: **Runtime → Restart session** first). ~15 min. It prints the comparison table and downloads `t2v_rowmap.pt` and `t2v_rowmap_results.json`.

In [ ]:
#@title **T2V-1K upgrade test: image-supervised row map** (Colab, T4 GPU, ~15 min)
# Rows of the classifier for classes the generator never saw are today predicted from text by a ridge
# map fitted to 897 zoo rows. Here the same text -> row map is trained on the photos of the 900 seen
# classes (episodic 10-way cross-entropy under the generated body), so it learns what makes a row
# classify, not only what a row looks like. Everything is chosen by cross-validation on seen classes;
# the official test tasks (zoo.make_tasks, seed 0, 100 per split) are scored once at the end.
import os, sys, subprocess, json, random, time, types, warnings
os.environ.update(TRANSFORMERS_VERBOSITY="error", HF_HUB_DISABLE_PROGRESS_BARS="1", TOKENIZERS_PARALLELISM="false")
warnings.filterwarnings("ignore")
SMOKE = os.environ.get("SMOKE") == "1"          # tiny local check of the code path only
REPO_DIR = os.environ.get("REPO_DIR", "/content/t2v-text-to-vision")
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/kalidolinux-sudo/t2v-text-to-vision", REPO_DIR], check=True)
elif not SMOKE:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "-q"], check=False)   # latest code (restart the session after a pull)
os.chdir(REPO_DIR); sys.path.insert(0, REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--root-user-action=ignore", "pyarrow"], check=False)
import torch, torch.nn.functional as F
import transformers; transformers.logging.set_verbosity_error()
from app import RELEASE_URL, download
from t2v import target
from t2v.generate import load_generator, generate_flat, neighbor_rows
from t2v.neighbors import mix_rows
from t2v.text_embed import embed_names
from t2v.data import _load_imagenet, MEAN, STD
from t2v.zoo import make_tasks

T0 = time.time()
def log(*a):
    print(f"[{(time.time() - T0) / 60:4.1f} min]", *a, flush=True)

DEV = "cuda" if torch.cuda.is_available() and not SMOKE else "cpu"
CKPT = "checkpoints/t2v_1k.pt"
if not os.path.exists(CKPT):
    download(RELEASE_URL, CKPT)
gen, tok, ck = load_generator(CKPT, DEV)
names, seen = ck["names"], sorted(ck["train_classes"])
sp = json.load(open("data/imagenet_split.json"))
meta = {**sp, "holdout": sorted(sp["holdout_random"] + sp["holdout_hard"])}
log("device", torch.cuda.get_device_name(0) if DEV == "cuda" else "cpu", "| seen classes", len(seen))

# class conditions: the generator's own (prompt ensemble + the 5 TokenFactory descriptions per class),
# and names only (ablation: the map without TokenFactory)
E = embed_names(names, DEV, ensemble=True, descriptions=ck["descriptions"])
E0 = embed_names(names, DEV, ensemble=True)
T = ck["row_table"].float().to(DEV)                       # zoo-mean learned row of every seen class
# TokenFactory's visual questionnaire: 60 attributes per class, scored 0-3 (t2v/attributes.py)
CONDS = {"text": E}
if os.path.exists("data/attributes_imagenet.json"):
    aj = json.load(open("data/attributes_imagenet.json"))
    AT = torch.tensor([aj["scores"].get(n) or [0] * len(aj["attributes"]) for n in names], dtype=torch.float, device=DEV) / 3
    log(f"TokenFactory attributes: {len(aj['scores'])} classes x {len(aj['attributes'])} attributes")
    CONDS["text + TokenFactory attributes"] = torch.cat([E, 0.3 * AT], 1)
    CONDS["TokenFactory attributes only"] = 0.3 * AT
log("class embeddings ready")

# photos: train split, first PER images of each class; the zoo used the first 260
PER, FIT, CV_LO, CV_HI = (12, 8, 8, 12) if SMOKE else (350, 160, 300, 350)
tr = _load_imagenet("train", "data", PER)
val = _load_imagenet("test", "data", 260)                 # ImageNet validation, 50 per class (test photos)
y_all = tr["y"]
rank = torch.zeros_like(y_all)
cnt = {}
for i, c in enumerate(y_all.tolist()):
    rank[i] = cnt.get(c, 0); cnt[c] = rank[i].item() + 1
is_seen = torch.isin(y_all, torch.tensor(seen))
keep = is_seen & ((rank < FIT) | ((rank >= CV_LO) & (rank < CV_HI)))
X, Y, CVM = tr["x"][keep], y_all[keep].to(DEV), ((rank[keep] >= CV_LO)).to(DEV)
del tr
log(f"photos: {int((~CVM).sum())} for fitting, {int(CVM.sum())} for cross-validation (images {CV_LO}-{CV_HI - 1} of each class, never used by the zoo)")

# the body the rows will sit on: the mean of 8 generated bodies (the body barely depends on the task)
rng0 = random.Random(0)
flats = [generate_flat(gen, tok, rng0.sample(seen, 10), steps=5, body_noise=0.0, gate="auto")[0] for _ in range(8)]
net = target.load_flat(target.build(ck["arch"]), torch.stack(flats).mean(0).cpu()).to(DEV).eval()
with torch.no_grad():
    FX = torch.cat([net.layers(torch.relu(net.bn1(net.conv1((X[i:i + 4096].to(DEV).float() / 255 - MEAN.to(DEV)) / STD.to(DEV))))).mean((2, 3))
                    for i in range(0, len(X), 4096)])
del X
log("features", tuple(FX.shape))


def ridge(Emb, ids, lam=3.0):  # the shipped prior (lambda 3.0 was chosen by CV at training time)
    A, B = Emb[ids], T[ids]
    return torch.linalg.solve(A.T @ A + lam * torch.eye(A.shape[1], device=DEV), A.T @ B)


class RowMap(torch.nn.Module):
    def __init__(self, W0, hid=0):
        super().__init__()
        self.M = torch.nn.Parameter(W0.clone()); self.W0 = W0.clone(); self.hid = hid
        if hid:
            self.mlp = torch.nn.Sequential(torch.nn.Linear(W0.shape[0], hid), torch.nn.GELU(), torch.nn.Linear(hid, W0.shape[1]))
            torch.nn.init.zeros_(self.mlp[2].weight); torch.nn.init.zeros_(self.mlp[2].bias)

    def forward(self, e):
        r = e @ self.M
        return r + self.mlp(e) if self.hid else r


def fit_map(Emb, ids, cfg, seed=0):
    torch.manual_seed(seed)
    ids_t = torch.tensor(ids, device=DEV)
    m = torch.isin(Y, ids_t) & ~CVM
    f, pos = FX[m], torch.searchsorted(ids_t, Y[m])
    order = torch.argsort(pos, stable=True)
    counts = torch.bincount(pos, minlength=len(ids)); starts = torch.cumsum(counts, 0) - counts
    g = RowMap(ridge(Emb, ids), cfg.get("hid", 0)).to(DEV)
    opt = torch.optim.Adam(g.parameters(), lr=cfg.get("lr", 1e-3))
    B, K, S = 8, 10, 32
    lab = torch.arange(K, device=DEV).repeat_interleave(S).repeat(B)
    for step in range(cfg["steps"] if not SMOKE else 20):
        if cfg["mode"] == "episodic":
            cls = torch.rand(B, len(ids), device=DEV).argsort(1)[:, :K]                        # (B, K)
            r = (torch.rand(B, K, S, device=DEV) * counts[cls][..., None]).long()
            feats = f[order[starts[cls][..., None] + r]].view(B, K * S, -1)                  # (B, KS, 64)
            rows = g(Emb[ids_t[cls]])                                                        # (B, K, 65)
            logits = torch.einsum("bnf,bkf->bnk", feats, rows[..., :-1]) + rows[..., -1][:, None]
            loss = F.cross_entropy(logits.reshape(-1, K), lab)
        else:
            i = torch.randint(len(f), (2048,), device=DEV)
            rows = g(Emb[ids_t])
            loss = F.cross_entropy(f[i] @ rows[:, :-1].T + rows[:, -1], pos[i])
        loss = loss + cfg["mu"] * ((g.M - g.W0) ** 2).sum()
        opt.zero_grad(); loss.backward(); opt.step()
    return g.eval()


def acc_rows(rows, cls):  # 10-way accuracy of rows on the cross-validation photos of `cls`
    ct = torch.tensor(cls, device=DEV)
    m = torch.isin(Y, ct) & CVM
    lab = torch.searchsorted(ct.sort().values, Y[m])
    perm = ct.argsort()
    logits = FX[m] @ rows[perm, :-1].T + rows[perm, -1]
    return (logits.argmax(1) == lab).float().mean().item()


# ---------------------------------------------------------------- 1. cross-validation on seen classes
CONFIGS = {  # the first run of this notebook: mu 1e-3 beat 1e-4, full softmax and an MLP
    "episodic mu1e-3 2k": {"mode": "episodic", "mu": 1e-3, "steps": 2000},
    "episodic mu3e-3 2k": {"mode": "episodic", "mu": 3e-3, "steps": 2000},
}
CANDS = {f"{c} | {k}": (c, k) for c in CONDS for k in CONFIGS}
BETAS = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5]
store_val = gen.neighbors["store"]                        # TokenFactory analogies (validation + held-out classes)
NT_CV = 4 if SMOKE else 40
cv = {k: [] for k in ["ridge", "zoo rows (oracle-ish)"] + list(CANDS)}
cv_beta = {}
cv_nodesc = []
folds = 2 if SMOKE else 5
for fold in range(folds):
    te = [c for c in seen if c % 5 == fold]; trc = [c for c in seen if c % 5 != fold]
    Wr = ridge(E, trc)
    rng = random.Random(f"cv{fold}")
    tasks = [rng.sample(te, 10) for _ in range(NT_CV)]
    with torch.no_grad():
        cv["ridge"] += [acc_rows(E[c] @ Wr, c) for c in tasks]
        cv["zoo rows (oracle-ish)"] += [acc_rows(T[c], c) for c in tasks]
    maps = {}
    for name, (cn, kn) in CANDS.items():
        C = CONDS[cn]
        maps[name] = (fit_map(C, trc, CONFIGS[kn]), C)
        with torch.no_grad():
            cv[name] += [acc_rows(maps[name][0](C[c]), c) for c in tasks]
    log(f"fold {fold}: " + "  ".join(f"{k} {sum(v[-NT_CV:]) / NT_CV:.3f}" for k, v in cv.items()))
    # analogy blend on top of each map: the fold's classes are new, so only the other seen classes may be picked
    name2id = {names[i]: i for i in trc}
    for name, (g, C) in maps.items():
        with torch.no_grad():
            for b in BETAS:
                s = []
                for c in tasks:
                    r = g(C[c])
                    nb = [mix_rows(store_val.get(names[i]), T, name2id) for i in c]
                    r = torch.stack([(1 - b) * r[j] + b * nb[j] if nb[j] is not None else r[j] for j in range(10)])
                    s.append(acc_rows(r, c))
                cv_beta.setdefault(name, {}).setdefault(b, []).extend(s)
    g0 = fit_map(E0, trc, CONFIGS["episodic mu1e-3 2k"])
    with torch.no_grad():
        cv_nodesc += [acc_rows(g0(E0[c]), c) for c in tasks]

mean = lambda v: sum(v) / len(v)
cv_mean = {k: mean(v) for k, v in cv.items()}
best = max(CANDS, key=lambda k: cv_mean[k])
beta_scores = {b: mean(v) for b, v in cv_beta[best].items()}
beta = max(beta_scores, key=beta_scores.get)
print("\n=== cross-validation on seen classes treated as new (%d folds x %d tasks) ===" % (folds, NT_CV))
for k, v in cv_mean.items():
    print(f"  {k:52s} {v:.4f}")
print(f"  map without TokenFactory descriptions (episodic mu1e-3 2k): {mean(cv_nodesc):.4f}")
print(f"  best map: {best}; TokenFactory analogy blend on top, by beta: " + "  ".join(f"{b}: {s:.4f}" for b, s in beta_scores.items()))
print(f"  chosen beta = {beta}")

# ---------------------------------------------------------------- 2. final map on all seen classes
best_cond, best_cfg = CANDS[best]
cfg = CONFIGS[best_cfg]
CB = CONDS[best_cond]
G = fit_map(CB, seen, cfg)
G0 = fit_map(E0, seen, CONFIGS["episodic mu1e-3 2k"])
others = {cn: fit_map(CONDS[cn], seen, cfg) for cn in CONDS if cn != best_cond}   # for comparison
log("final maps fitted on all", len(seen), "seen classes")

# ---------------------------------------------------------------- 3. the official test tasks, scored once
val_x, val_y = val["x"], val["y"]
tnet = target.build(ck["arch"]).to(DEV).eval()


def acc_flat(flat, cls):
    m = torch.isin(val_y, torch.tensor(cls))
    x = (val_x[m].to(DEV).float() / 255 - MEAN.to(DEV)) / STD.to(DEV)
    lab = torch.tensor([cls.index(int(v)) for v in val_y[m]], device=DEV)
    target.load_flat(tnet, flat)
    with torch.no_grad():
        return (tnet(x)[:, :len(cls)].argmax(1) == lab).float().mean().item()


NT = 6 if SMOKE else 100
official, per_task = {}, {}
for split in ["eval_seen", "eval_unseen", "eval_novel", "eval_novel_hard"]:
    tasks = make_tasks(types.SimpleNamespace(split=split, seed=0, n=NT, unseen_frac=0.5, min_k=10, n_train_ref=1000), meta, 10)[:NT]
    res = {}
    for cls in tasks:
        torch.manual_seed(0)
        info = {}
        base = generate_flat(gen, tok, cls, steps=5, emb=E[cls], body_noise=0.0, gate="auto", info=info)[0]
        assert info["names"] == [names[c] for c in cls], (info["names"], cls)
        known = torch.tensor([info["known"][n] for n in info["names"]], device=DEV)
        rows = target.head_rows(ck["arch"], base)[:len(cls)]
        with torch.no_grad():
            nb, _ = neighbor_rows(gen, [names[c] for c in cls], E[cls], "llm", ~known)
            mr = G(CB[cls])
            variants = {f"best map ({best_cond})": mr, "map: names only (no TokenFactory)": G0(E0[cls])}
            for cn, g in others.items():
                variants[f"map: {cn}"] = g(CONDS[cn][cls])
            if nb is not None:
                has = ~nb.isnan().any(1)
                variants[f"best map + TokenFactory analogies (beta {beta})"] = torch.where(has[:, None], (1 - beta) * mr + beta * nb.nan_to_num(), mr)
            else:
                variants[f"best map + TokenFactory analogies (beta {beta})"] = mr
        res.setdefault("shipped T2V-1K", []).append(acc_flat(base, cls))
        for k, new in variants.items():
            r2 = rows.clone(); r2[~known] = new[~known]
            res.setdefault(k, []).append(acc_flat(target.add_head_rows(ck["arch"], base, r2 - rows), cls))
    official[split] = {k: mean(v) for k, v in res.items()}
    per_task[split] = res
    log(split, "  ".join(f"{k} {v:.3f}" for k, v in official[split].items()))

print("\n" + "=" * 96)
print(f"T2V-1K + image-supervised row map | official test tasks, {NT} per split, 10 classes, ImageNet validation photos")
cols = ["eval_seen", "eval_unseen", "eval_novel", "eval_novel_hard"]
print(f"{'':62s}" + "".join(f"{c:>11s}" for c in ["seen", "half new", "all new", "hard"]))
for k in official["eval_novel"]:
    print(f"{k:62s}" + "".join(f"{official[c][k]:11.3f}" for c in cols))
final = f"best map + TokenFactory analogies (beta {beta})"
for c in cols:
    d = torch.tensor(per_task[c][final]) - torch.tensor(per_task[c]["shipped T2V-1K"])
    print(f"  {c:16s} upgrade - shipped: {d.mean().item():+.4f} ± {(d.std() / len(d) ** 0.5).item():.4f} (s.e.)  wins {int((d > 0).sum())}  losses {int((d < 0).sum())}")
print("=" * 96)

out = {"config": best, "cfg": cfg, "beta": beta, "cv": cv_mean, "cv_beta": beta_scores, "cv_names_only": mean(cv_nodesc),
       "official": official, "per_task": per_task, "tasks_per_split": NT}
torch.save({"M": G.M.detach().cpu(), "hid": G.hid, "mlp": G.mlp.state_dict() if G.hid else None, "beta": beta,
            "config": best, "condition": best_cond, "attribute_scale": 0.3, "cv": cv_mean}, "/content/t2v_rowmap.pt" if not SMOKE else "rowmap_smoke.pt")
json.dump(out, open("/content/t2v_rowmap_results.json" if not SMOKE else "rowmap_smoke.json", "w"), indent=1)
log("saved t2v_rowmap.pt and t2v_rowmap_results.json")
if not SMOKE:
    try:
        from google.colab import files
        files.download("/content/t2v_rowmap_results.json"); files.download("/content/t2v_rowmap.pt")
    except Exception:
        pass
